# تبدیلات آفین / Affine Transformations

**هدف:** یادگیری و پیاده‌سازی تبدیلات آفین شامل انتقال، چرخش، مقیاس‌بندی و برش

**Objective:** Learn and implement affine transformations including translation, rotation, scaling, and shearing

---

## محتوا / Contents:
1. مقدمه‌ای بر تبدیلات آفین / Introduction to Affine Transformations
2. انتقال / Translation
3. چرخش / Rotation
4. مقیاس‌بندی / Scaling
5. برش / Shearing
6. ترکیب تبدیلات / Combining Transformations
7. کاربردهای عملی / Practical Applications

In [ ]:
# Import libraries
import cv2
import numpy as np
import matplotlib.pyplot as plt
from typing import Tuple

# تنظیمات نمایش / Display settings
plt.rcParams['figure.figsize'] = (15, 10)
plt.rcParams['font.size'] = 12

print(f"OpenCV version: {cv2.__version__}")
print(f"NumPy version: {np.__version__}")

## 1. توابع کمکی / Helper Functions

In [ ]:
def show_comparison(images: list, titles: list, rows: int = 1, cols: int = 2, cmap: str = 'gray'):
    """
    نمایش مقایسه چند تصویر
    
    Args:
        images: لیست تصاویر
        titles: عناوین تصاویر
        rows: تعداد ردیف‌ها
        cols: تعداد ستون‌ها
        cmap: نقشه رنگی
    """
    fig, axes = plt.subplots(rows, cols, figsize=(15, 5*rows))
    axes = axes.flatten() if rows * cols > 1 else [axes]
    
    for idx, (img, title) in enumerate(zip(images, titles)):
        if len(img.shape) == 2:
            axes[idx].imshow(img, cmap=cmap)
        else:
            axes[idx].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        axes[idx].set_title(title, fontsize=14)
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.show()


def create_test_image(size: Tuple[int, int] = (400, 400)) -> np.ndarray:
    """
    ایجاد تصویر تست با اشکال هندسی
    
    Args:
        size: اندازه تصویر (ارتفاع، عرض)
    
    Returns:
        تصویر تست
    """
    # ایجاد تصویر سفید
    img = np.ones((size[0], size[1], 3), dtype=np.uint8) * 255
    
    # رسم مستطیل آبی
    cv2.rectangle(img, (50, 50), (150, 150), (255, 0, 0), -1)
    
    # رسم دایره قرمز
    cv2.circle(img, (250, 100), 50, (0, 0, 255), -1)
    
    # رسم مثلث سبز
    pts = np.array([[100, 250], [150, 350], [50, 350]], np.int32)
    cv2.fillPoly(img, [pts], (0, 255, 0))
    
    # نوشتن متن
    cv2.putText(img, 'OpenCV', (200, 300), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 0), 2)
    
    return img


def draw_grid(img: np.ndarray, step: int = 50, color: Tuple[int, int, int] = (200, 200, 200)):
    """
    رسم شبکه روی تصویر برای نمایش بهتر تبدیلات
    
    Args:
        img: تصویر ورودی
        step: فاصله خطوط شبکه
        color: رنگ خطوط
    
    Returns:
        تصویر با شبکه
    """
    img_grid = img.copy()
    h, w = img.shape[:2]
    
    # خطوط عمودی
    for x in range(0, w, step):
        cv2.line(img_grid, (x, 0), (x, h), color, 1)
    
    # خطوط افقی
    for y in range(0, h, step):
        cv2.line(img_grid, (0, y), (w, y), color, 1)
    
    return img_grid

## 2. ایجاد تصویر تست / Create Test Image

In [ ]:
# ایجاد تصویر تست
test_img = create_test_image()
test_img_grid = draw_grid(test_img)

show_comparison([test_img, test_img_grid], 
                ['تصویر اصلی / Original Image', 'تصویر با شبکه / Image with Grid'],
                rows=1, cols=2)

## 3. انتقال (Translation)

**توضیح:** انتقال یعنی جابجایی تصویر در جهت x و y. ماتریس تبدیل:

$$M = \begin{bmatrix} 1 & 0 & t_x \\ 0 & 1 & t_y \end{bmatrix}$$

**Explanation:** Translation means shifting the image in x and y directions.

In [ ]:
def translate_image(img: np.ndarray, tx: float, ty: float) -> np.ndarray:
    """
    انتقال تصویر
    
    Args:
        img: تصویر ورودی
        tx: جابجایی در جهت x
        ty: جابجایی در جهت y
    
    Returns:
        تصویر جابجا شده
    """
    # ایجاد ماتریس تبدیل
    M = np.float32([[1, 0, tx],
                    [0, 1, ty]])
    
    # اعمال تبدیل
    h, w = img.shape[:2]
    translated = cv2.warpAffine(img, M, (w, h))
    
    return translated


# مثال: جابجایی 50 پیکسل به راست و 30 پیکسل به پایین
translated_img = translate_image(test_img, 50, 30)

show_comparison([test_img, translated_img],
                ['تصویر اصلی / Original', 'جابجا شده (50, 30) / Translated (50, 30)'],
                rows=1, cols=2)

print("ماتریس تبدیل انتقال / Translation Matrix:")
print(np.float32([[1, 0, 50], [0, 1, 30]]))

## 4. چرخش (Rotation)

**توضیح:** چرخش تصویر حول یک نقطه (معمولاً مرکز). OpenCV تابع `getRotationMatrix2D` را برای محاسبه ماتریس چرخش فراهم می‌کند.

**Explanation:** Rotating the image around a point (usually the center).

In [ ]:
def rotate_image(img: np.ndarray, angle: float, center: Tuple[int, int] = None, scale: float = 1.0) -> np.ndarray:
    """
    چرخش تصویر
    
    Args:
        img: تصویر ورودی
        angle: زاویه چرخش (درجه، خلاف جهت عقربه‌های ساعت)
        center: مرکز چرخش (اگر None باشد، مرکز تصویر استفاده می‌شود)
        scale: ضریب مقیاس
    
    Returns:
        تصویر چرخیده
    """
    h, w = img.shape[:2]
    
    # اگر مرکز مشخص نشده، مرکز تصویر را استفاده کن
    if center is None:
        center = (w // 2, h // 2)
    
    # محاسبه ماتریس چرخش
    M = cv2.getRotationMatrix2D(center, angle, scale)
    
    # اعمال تبدیل
    rotated = cv2.warpAffine(img, M, (w, h))
    
    return rotated, M


# مثال: چرخش 45 درجه
rotated_45, M_45 = rotate_image(test_img, 45)
rotated_90, M_90 = rotate_image(test_img, 90)
rotated_neg30, M_neg30 = rotate_image(test_img, -30)

show_comparison([test_img, rotated_45, rotated_90, rotated_neg30],
                ['اصلی / Original', 'چرخش 45° / Rotated 45°', 
                 'چرخش 90° / Rotated 90°', 'چرخش -30° / Rotated -30°'],
                rows=2, cols=2)

print("ماتریس چرخش 45 درجه / Rotation Matrix (45°):")
print(M_45)

## 5. مقیاس‌بندی (Scaling)

**توضیح:** تغییر اندازه تصویر. می‌توان از `cv2.resize()` یا `cv2.warpAffine()` استفاده کرد.

**Explanation:** Changing the size of the image.

In [ ]:
# روش 1: استفاده از cv2.resize()
h, w = test_img.shape[:2]

# بزرگ‌تر کردن (2 برابر)
scaled_up = cv2.resize(test_img, None, fx=2.0, fy=2.0, interpolation=cv2.INTER_LINEAR)

# کوچک‌تر کردن (0.5 برابر)
scaled_down = cv2.resize(test_img, None, fx=0.5, fy=0.5, interpolation=cv2.INTER_AREA)

# مقیاس‌بندی غیریکنواخت
scaled_non_uniform = cv2.resize(test_img, None, fx=1.5, fy=0.7, interpolation=cv2.INTER_LINEAR)

print(f"اندازه اصلی / Original size: {test_img.shape[:2]}")
print(f"بزرگ‌تر شده / Scaled up: {scaled_up.shape[:2]}")
print(f"کوچک‌تر شده / Scaled down: {scaled_down.shape[:2]}")
print(f"مقیاس غیریکنواخت / Non-uniform: {scaled_non_uniform.shape[:2]}")

# نمایش (تنظیم اندازه برای نمایش)
scaled_up_display = cv2.resize(scaled_up, (400, 400))
scaled_down_display = cv2.resize(scaled_down, (400, 400))
scaled_non_uniform_display = cv2.resize(scaled_non_uniform, (400, 400))

show_comparison([test_img, scaled_up_display, scaled_down_display, scaled_non_uniform_display],
                ['اصلی / Original', 'بزرگ‌تر (2x) / Scaled Up (2x)',
                 'کوچک‌تر (0.5x) / Scaled Down (0.5x)', 'غیریکنواخت (1.5x, 0.7x) / Non-uniform'],
                rows=2, cols=2)

In [ ]:
# روش 2: استفاده از warpAffine با ماتریس مقیاس‌بندی
def scale_with_affine(img: np.ndarray, sx: float, sy: float) -> np.ndarray:
    """
    مقیاس‌بندی با استفاده از تبدیل آفین
    
    Args:
        img: تصویر ورودی
        sx: ضریب مقیاس در جهت x
        sy: ضریب مقیاس در جهت y
    
    Returns:
        تصویر مقیاس‌بندی شده
    """
    h, w = img.shape[:2]
    
    # ماتریس مقیاس‌بندی
    M = np.float32([[sx, 0, 0],
                    [0, sy, 0]])
    
    # محاسبه اندازه جدید
    new_w = int(w * sx)
    new_h = int(h * sy)
    
    scaled = cv2.warpAffine(img, M, (new_w, new_h))
    
    return scaled


scaled_affine = scale_with_affine(test_img, 1.5, 1.5)
print(f"مقیاس‌بندی با آفین / Affine scaling: {scaled_affine.shape[:2]}")

## 6. برش (Shearing)

**توضیح:** برش باعث کج شدن تصویر می‌شود. ماتریس برش:

$$M = \begin{bmatrix} 1 & sh_x & 0 \\ sh_y & 1 & 0 \end{bmatrix}$$

**Explanation:** Shearing skews the image.

In [ ]:
def shear_image(img: np.ndarray, shx: float, shy: float) -> np.ndarray:
    """
    برش تصویر
    
    Args:
        img: تصویر ورودی
        shx: ضریب برش در جهت x
        shy: ضریب برش در جهت y
    
    Returns:
        تصویر برش خورده
    """
    h, w = img.shape[:2]
    
    # ماتریس برش
    M = np.float32([[1, shx, 0],
                    [shy, 1, 0]])
    
    # محاسبه اندازه جدید برای جلوگیری از بریدن
    new_w = int(w + abs(shx * h))
    new_h = int(h + abs(shy * w))
    
    sheared = cv2.warpAffine(img, M, (new_w, new_h))
    
    return sheared


# برش افقی
sheared_x = shear_image(test_img, 0.3, 0)

# برش عمودی
sheared_y = shear_image(test_img, 0, 0.3)

# برش در هر دو جهت
sheared_both = shear_image(test_img, 0.2, 0.2)

# تنظیم اندازه برای نمایش
sheared_x_display = cv2.resize(sheared_x, (400, 400))
sheared_y_display = cv2.resize(sheared_y, (400, 400))
sheared_both_display = cv2.resize(sheared_both, (400, 400))

show_comparison([test_img, sheared_x_display, sheared_y_display, sheared_both_display],
                ['اصلی / Original', 'برش افقی / Horizontal Shear',
                 'برش عمودی / Vertical Shear', 'برش دوطرفه / Both Directions'],
                rows=2, cols=2)

## 7. ترکیب تبدیلات (Combining Transformations)

**توضیح:** می‌توان چندین تبدیل را با ضرب ماتریس‌ها ترکیب کرد. توجه: ترتیب مهم است!

**Explanation:** Multiple transformations can be combined by multiplying matrices. Note: Order matters!

In [ ]:
# مثال: چرخش و سپس انتقال
h, w = test_img.shape[:2]
center = (w // 2, h // 2)

# ماتریس چرخش
M_rot = cv2.getRotationMatrix2D(center, 30, 1.0)

# ماتریس انتقال
M_trans = np.float32([[1, 0, 50],
                      [0, 1, 30]])

# ترکیب: ابتدا چرخش، سپس انتقال
# برای ترکیب، باید ماتریس‌ها را به فرم همگن تبدیل کنیم
M_rot_homogeneous = np.vstack([M_rot, [0, 0, 1]])
M_trans_homogeneous = np.vstack([M_trans, [0, 0, 1]])

# ضرب ماتریس‌ها (از راست به چپ)
M_combined = M_trans_homogeneous @ M_rot_homogeneous
M_combined = M_combined[:2, :]  # برگشت به فرم 2x3

# اعمال تبدیل ترکیبی
combined = cv2.warpAffine(test_img, M_combined, (w, h))

# مقایسه با اعمال جداگانه
step1 = cv2.warpAffine(test_img, M_rot, (w, h))
step2 = cv2.warpAffine(step1, M_trans, (w, h))

show_comparison([test_img, step1, step2, combined],
                ['اصلی / Original', 'گام 1: چرخش / Step 1: Rotation',
                 'گام 2: انتقال / Step 2: Translation', 'ترکیبی / Combined'],
                rows=2, cols=2)

print("ماتریس ترکیبی / Combined Matrix:")
print(M_combined)

## 8. کاربرد عملی: افزایش داده (Data Augmentation)

**توضیح:** تبدیلات آفین برای افزایش داده در یادگیری ماشین استفاده می‌شوند.

**Explanation:** Affine transformations are used for data augmentation in machine learning.

In [ ]:
def augment_image(img: np.ndarray, num_augmentations: int = 5) -> list:
    """
    ایجاد نسخه‌های افزوده از تصویر
    
    Args:
        img: تصویر ورودی
        num_augmentations: تعداد نسخه‌های افزوده
    
    Returns:
        لیست تصاویر افزوده شده
    """
    h, w = img.shape[:2]
    center = (w // 2, h // 2)
    augmented = []
    
    for i in range(num_augmentations):
        # پارامترهای تصادفی
        angle = np.random.uniform(-30, 30)
        scale = np.random.uniform(0.8, 1.2)
        tx = np.random.randint(-50, 50)
        ty = np.random.randint(-50, 50)
        
        # چرخش و مقیاس
        M_rot = cv2.getRotationMatrix2D(center, angle, scale)
        
        # انتقال
        M_rot[0, 2] += tx
        M_rot[1, 2] += ty
        
        # اعمال تبدیل
        aug_img = cv2.warpAffine(img, M_rot, (w, h))
        augmented.append(aug_img)
    
    return augmented


# ایجاد 6 نسخه افزوده
augmented_images = augment_image(test_img, 6)

# نمایش
all_images = [test_img] + augmented_images[:5]
titles = ['اصلی / Original'] + [f'افزوده {i+1} / Aug {i+1}' for i in range(5)]

show_comparison(all_images, titles, rows=2, cols=3)

## 9. تمرین‌ها / Exercises

### تمرین 1 / Exercise 1:
تصویری را 180 درجه بچرخانید و سپس آن را 100 پیکسل به چپ و 50 پیکسل به بالا جابجا کنید.

Rotate an image 180 degrees and then translate it 100 pixels left and 50 pixels up.

### تمرین 2 / Exercise 2:
تابعی بنویسید که تصویر را حول گوشه بالا-چپ بچرخاند (نه مرکز).

Write a function that rotates an image around the top-left corner (not the center).

### تمرین 3 / Exercise 3:
یک تبدیل آفین سفارشی ایجاد کنید که ترکیبی از چرخش 45 درجه، مقیاس 1.5 و برش 0.2 باشد.

Create a custom affine transformation that combines 45-degree rotation, 1.5x scaling, and 0.2 shearing.

### تمرین 4 / Exercise 4:
تابع افزایش داده را تغییر دهید تا flip افقی و عمودی را نیز شامل شود.

Modify the data augmentation function to also include horizontal and vertical flips.

In [ ]:
# فضای کار برای تمرین‌ها / Workspace for exercises

# تمرین 1 / Exercise 1
# کد خود را اینجا بنویسید


# تمرین 2 / Exercise 2
# کد خود را اینجا بنویسید


# تمرین 3 / Exercise 3
# کد خود را اینجا بنویسید


# تمرین 4 / Exercise 4
# کد خود را اینجا بنویسید